In [13]:
import sys
from contextlib import redirect_stderr
from datetime import timedelta
from shutil import which

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# MCP SDK 在导入时记录子进程的默认 stderr，该流需要支持 fileno()。
with redirect_stderr(sys.__stderr__):
    from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

True

In [14]:
npx = which("npx.cmd" if sys.platform == "win32" else "npx")
if npx is None:
    raise RuntimeError("找不到 npx。请安装 Node.js 20 或更新版本，并重启 Jupyter 内核。")

client = MultiServerMCPClient(
    {
        "time-mcp": {
            "transport": "stdio",
            "command": npx,
            "args": ["-y", "time-mcp"],
            "session_kwargs": {"read_timeout_seconds": timedelta(seconds=150)},
        },
        "12306": {
            "transport": "streamable_http",
            "url": "https://mcp.api-inference.modelscope.net/5fc510beb7944a/mcp",
            "session_kwargs": {"read_timeout_seconds": timedelta(seconds=30)},
        },
    }
)

print("正在加载 MCP 工具……", flush=True)
tools = await client.get_tools()
print([tool.name for tool in tools])

正在加载 MCP 工具……
['current_time', 'relative_time', 'days_in_month', 'get_timestamp', 'convert_time', 'get_week_year', 'get-current-date', 'get-stations-code-in-city', 'get-station-code-of-citys', 'get-station-code-by-names', 'get-station-by-telecode', 'get-tickets', 'get-interline-tickets', 'get-train-route-stations']


In [15]:
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}},
)
agent = create_agent(model=model, tools=tools)

response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下明天从重庆到成都的高铁")]}
)

In [16]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

帮我查一下明天从重庆到成都的高铁
================================== Ai Message ==================================

我来帮您查询明天从重庆到成都的高铁票。
Tool Calls:
  get-current-date (call_00_glsXCfM0j6QdiL0R7RhT7100)
 Call ID: call_00_glsXCfM0j6QdiL0R7RhT7100
  Args:
================================= Tool Message =================================
Name: get-current-date

[{'type': 'text', 'text': '2026-11-20', 'id': 'lc_3a7d61ef-b8d0-4960-85c6-c7fde1f298d7'}]
================================== Ai Message ==================================
Tool Calls:
  get-tickets (call_00_dDNkGgBYHdOzzozEOfCb9963)
 Call ID: call_00_dDNkGgBYHdOzzozEOfCb9963
  Args:
    date: 2026-11-21
    fromStation: 重庆
    toStation: 成都
    trainFilterFlags: G
================================= Tool Message =================================
Name: get-tickets

[{'type': 'text', 'text': '车次|出发站 -> 到达站|出发时间 -> 到达时间|历时\nG8658 重庆西(telecode:CXW) -> 成都东(telecode:ICW) 00:29 ->